In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def is_amharic(text):
    return any('\u1200' <= ch <= '\u137F' for ch in text)

def scrape_chapter(book_id, book_chapter):
    url = f"http://bibleonline.geezexperience.com/amharic/list.php?book={book_id}&chapter={book_chapter}"

    try:
        response = requests.get(url, timeout=20 )
        # the 20 sec show that it waits for 20 sec before giving up
        response.encoding = "utf-8"
    except Exception as e:
        print(f"  Request failed: {e}")
        return []

    soup = BeautifulSoup(response.text, "html.parser") # tring to change to soup object

    # Remove noisy sections
    for tag in soup.find_all(["nav", "footer", "script", "style"]): ## trying to select content
        tag.decompose()
    for ul in soup.find_all("ul"):
        if any(kw in ul.get_text() for kw in ["ብሉይ", "አዲስ", "Related"]):
            ul.decompose()

    lines = [l.strip() for l in soup.get_text(separator="\n").split("\n") if l.strip()]

    verses = []
    i = 0

    def is_verse_num(s):
        return s.isdigit() and 1 <= int(s) <= 200

    # Skip header/nav lines until first verse number
    while i < len(lines):
        if is_verse_num(lines[i]):
            break
        i += 1

    # Parse verse blocks
    while i < len(lines):
        line = lines[i]

        if not is_verse_num(line):
            i += 1
            continue

        verse_num = int(line)
        i += 1

        # Collect Amharic text
        amharic_parts = []
        while i < len(lines) and not is_verse_num(lines[i]):
            candidate = lines[i]
            if candidate.startswith("http"):
                i += 1
                continue
            if is_amharic(candidate):
                amharic_parts.append(candidate)
            else:
                break
            i += 1

        # Collect English text
        english_parts = []
        while i < len(lines) and not is_verse_num(lines[i]):
            candidate = lines[i]
            if candidate.startswith("http"):
                i += 1
                continue
            if any(kw in candidate for kw in ["Back to top", "Bootstrap", "geezexperience", "Contact"]):
                break
            if not is_amharic(candidate) and candidate[:1].isalpha():
                english_parts.append(candidate)
            else:
                break
            i += 1

        amharic_text = " ".join(amharic_parts).strip()
        english_text = " ".join(english_parts).strip()

        if amharic_text or english_text:
            verses.append({
                "verse":        verse_num,
                "amharic_text": amharic_text,
                "english_text": english_text,
            })

    return verses


def get_book_name(book_id):
    """Scrape the book name from the first chapter page heading."""
    url = f"http://bibleonline.geezexperience.com/amharic/list.php?book={book_id}&chapter=1"
    try:
        response = requests.get(url, timeout=20)
        response.encoding = "utf-8"
        soup = BeautifulSoup(response.text, "html.parser")
        h1 = soup.find("h1")
        if h1:
            # heading format: "ዘፍጥረት 1 - Genesis 1"
            parts = h1.get_text().split(" - ")
            amharic_name = parts[0].replace("1", "").strip() if parts else ""
            english_name = parts[1].replace("1", "").strip() if len(parts) > 1 else ""
            return amharic_name, english_name
    except:
        pass
    return "", ""


def main():
    all_rows = []

    for book_id in range(1, 67):          # loop all 66 books
        amharic_name, english_name = get_book_name(book_id)
        print(f"\nBook {book_id}/66: {english_name}")

        book_chapter = 1
        while True:                        # loop chapters until empty
            verses = scrape_chapter(book_id, book_chapter)

            if not verses:                 # no verses = no more chapters
                print(f"  → {book_chapter - 1} chapters scraped")
                break

            print(f"  ch.{book_chapter} ({len(verses)} verses)", end="  ", flush=True)

            for v in verses:
                all_rows.append({
                    "book_id":      book_id,
                    "book_amharic": amharic_name,
                    "book_english": english_name,
                    "chapter":      book_chapter,
                    "verse":        v["verse"],
                    "amharic_text": v["amharic_text"],
                    "english_text": v["english_text"],
                })

            book_chapter += 1
            # time.sleep(0.4)            

    # Save to CSV
    df = pd.DataFrame(all_rows)
    df.to_csv("amharic_bible.csv", index=False, encoding="utf-8-sig")

    print(f"\n\n Done!")
    print(f"   Total verses  : {len(df):,}")
    print(f"   Total books   : {df['book_english'].nunique()}")
    print(f"   Saved to      : amharic_bible.csv")


if __name__ == "__main__":
    main()